# Homework 5.1: Reinforcement Learning with Verifiable Rewards, from scratch

In HW3 you built language models that learn by **imitation**: next-token prediction on text
written by somebody else.  In this notebook you will take a tiny pretrained language model and
improve it with **reinforcement learning from a verifiable reward (RLVR)**, using
**GRPO** (Group Relative Policy Optimization), the algorithm used to train DeepSeek-R1.

Everything here is small enough to run on a laptop CPU in a few minutes, so that every line
of the RL loop is visible.  Our overall goals are to:
- See the full RLVR loop: **sample** completions, **verify** them, compute **advantages**, take a **policy-gradient** step.
- Understand what GRPO changes relative to PPO (no learned value function; the group is the baseline).
- Measure what RL does and does not do to a model (pass@1 versus pass@k, output format, output length).
- Experience **reward hacking** first-hand by training against a sloppy verifier.

> **Key Learning Objectives:**
> 1. **The verifier.**  Write the reward function $r(x, y) \in \{0, 1\}$ that checks a completion against ground truth.
> 2. **Group-relative advantages.**  Given $G$ sampled completions $y_1 \dots y_G$ for the same prompt with rewards $r_1 \dots r_G$, compute
> $$A_i = \frac{r_i - \mathrm{mean}(r_{1..G})}{\mathrm{std}(r_{1..G}) + \epsilon}.$$
> 3. **The GRPO loss.**  Implement the policy-gradient loss with an optional KL penalty to the base model:
> $$\mathcal{L} = -\frac{1}{G}\sum_{i=1}^{G} A_i \cdot \frac{1}{|y_i|}\sum_{t=1}^{|y_i|} \log \pi_\theta(y_{i,t} \mid x, y_{i,<t}) \;+\; \beta\, \mathrm{KL}\!\left(\pi_\theta \,\|\, \pi_{\mathrm{ref}}\right).$$
> 4. **Experiments.**  Run the ablations at the end and explain what you see.

**Your Task:**
* Implement the **TODO** in `extract_answer()` / `reward_fn()` (the verifier).
* Implement the **TODO** in `group_advantages()`.
* Implement the **TODO** in `grpo_loss()`.
* Run the experiments in the *Task for You* section and answer the questions in the markdown cells.

Let's get started with the necessary imports and random seeds.

In [ ]:
# Imports and random seed setup
import copy
import math
import random
import re
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

torch.manual_seed(0)   # Ensures that the PyTorch random generators are deterministic
random.seed(0)
torch.set_num_threads(max(1, torch.get_num_threads()))
print("torch", torch.__version__)

## The task: sum three digits

We need a task where (a) a tiny model can learn something, and (b) a program can check the answer.
Our task is adding three single digits:

```
3+8+6=   →   17
```

A prompt is always six characters (`a+b+c=`), and the model must produce the answer followed by a
period, which we use as the end-of-answer token.  The vocabulary is just the digits, `+`, `=`, `.`,
and a padding symbol `_`.

In [ ]:
# Tokenizer: one token per character
VOCAB = list("0123456789+=.") + ["_"]      # "." ends an answer, "_" is padding
stoi = {ch: i for i, ch in enumerate(VOCAB)}
itos = {i: ch for ch, i in stoi.items()}
PAD, EOS = stoi["_"], stoi["."]
PROMPT_LEN = 6          # every prompt looks like "a+b+c=" (6 characters)
MAX_NEW = 10            # longest completion we allow, e.g. "11+6=17." is 8 characters

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids if int(i) != PAD)

def make_problem(rng):
    a, b, c = rng.randint(0, 9), rng.randint(0, 9), rng.randint(0, 9)
    return f"{a}+{b}+{c}=", a + b + c

rng = random.Random(0)
for _ in range(3):
    print(make_problem(rng))

## A noisy teacher writes the pretraining data

Real language models are pretrained on internet text that contains both correct and incorrect
solutions, written in many styles.  We imitate that with a **noisy teacher** that writes each
solution in one of two styles, chosen by a coin flip:

| style | example | teacher is correct |
|---|---|---|
| **direct** | `3+8+6=17.` | 50% of the time |
| **scratchpad** | `3+8+6=11+6=17.` (writes the partial sum first) | 90% of the time |

When the teacher is wrong, its answer is off by a plausible amount ($\pm1$, $\pm2$, or $\pm10$).

A model that imitates this teacher perfectly will be right only about 70% of the time when it
samples an answer.  Notice that the *knowledge* of how to add is fully present in the data:
the scratchpad style is nearly always right.  The question RL will answer is whether we can get
the model to *use* what it knows.

In [ ]:
def corrupt(x, rng):
    '''Return a plausible *wrong* answer near x.'''
    while True:
        y = x + rng.choice([-10, -2, -1, 1, 2, 10])
        if 0 <= y <= 30 and y != x:
            return y

def teacher_completion(prompt, answer, rng, p_scratch=0.5, p_direct_ok=0.5, p_scratch_ok=0.9):
    '''Write a solution in the scratchpad style (probability p_scratch) or the direct style.'''
    a, b, c = (int(x) for x in prompt[:-1].split("+"))
    if rng.random() < p_scratch:
        final = answer if rng.random() < p_scratch_ok else corrupt(answer, rng)
        return f"{a + b}+{c}={final}."
    final = answer if rng.random() < p_direct_ok else corrupt(answer, rng)
    return f"{final}."

def make_corpus(n, seed=0, **teacher_kw):
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        p, ans = make_problem(rng)
        out.append(p + teacher_completion(p, ans, rng, **teacher_kw))
    return out

corpus = make_corpus(20000)
print("\n".join(corpus[:8]))

## The verifier

RLVR needs a program that says whether a completion is right.  Ours reads the **final number**:
the digits between the last `=` (or the start of the completion, in the direct style) and the
first `.`.  If there is no well-formed number there, the completion is wrong.

**TODO:** Implement `extract_answer()` and `reward_fn()`.

*Hints:*
- `completion.split(".")[0]` gives the text before the first period.
- Splitting that on `"="` and taking the last piece gives the candidate answer.
- Use `str.isdigit()` to decide whether the candidate is a number; return `None` if it is not.
- The reward is `1.0` when the extracted answer equals `a + b + c`, else `0.0`.

In [ ]:
def extract_answer(completion):
    '''Return the final number in a completion as an int, or None if there is no well-formed number.'''
    # ----- TODO: Student Implementation Starts Here -----
    # Steps:
    #   1. body = everything before the first "."      e.g. "11+6=17."  ->  "11+6=17"
    #   2. candidate = the text after the last "="       e.g. "11+6=17"   ->  "17"   (a direct answer "17" has no "=")
    #   3. return int(candidate) if it is all digits, otherwise None       e.g. "1x" -> None,  "" -> None
    raise NotImplementedError("Implement extract_answer")
    # ----- TODO: Student Implementation Ends Here -----

def is_scratchpad(completion):
    '''True if the completion shows a partial sum before the final answer (used for analysis only).'''
    return "=" in completion.split(".")[0]

def reward_fn(prompt, completion):
    '''Verifiable reward: 1.0 if the completion's final number equals the true sum, else 0.0.'''
    a, b, c = (int(x) for x in prompt[:-1].split("+"))
    # ----- TODO: Student Implementation Starts Here -----
    # r(x, y) = 1 if extract_answer(y) == a + b + c else 0.   Note that None never equals an int.
    raise NotImplementedError("Implement reward_fn")
    # ----- TODO: Student Implementation Ends Here -----

# Sanity checks
assert extract_answer("17.") == 17
assert extract_answer("11+6=17.") == 17
assert extract_answer("11+6=1x.") is None
assert extract_answer("") is None
assert reward_fn("3+8+6=", "17.") == 1.0
assert reward_fn("3+8+6=", "11+6=17.") == 1.0
assert reward_fn("3+8+6=", "11+6=18.") == 0.0
assert reward_fn("3+8+6=", "17+0.") == 0.0
print("verifier OK")

teacher_acc = sum(reward_fn(s[:PROMPT_LEN], s[PROMPT_LEN:]) for s in corpus[:2000]) / 2000
print(f"teacher accuracy on its own corpus: {teacher_acc:.3f}")

## A tiny GPT

The policy is a two-layer transformer with about 100k parameters, the same architecture you built
in HW3 (causal self-attention followed by an MLP, with pre-layer-norm and residual connections).
Nothing here needs to change.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        T = x.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=causal, need_weights=False)
        x = x + a
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self, vocab_size=len(VOCAB), d_model=64, n_heads=4, n_layers=2, max_len=PROMPT_LEN + MAX_NEW):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList(Block(d_model, n_heads) for _ in range(n_layers))
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        T = idx.size(1)
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln(x))

print("parameters:", sum(p.numel() for p in TinyGPT().parameters()))

## Stage 1: pretraining by imitation

First we train the model the ordinary way, with next-token cross-entropy on the teacher's text.
The result is our **base model** $\pi_{\mathrm{ref}}$.  It takes about 20 seconds on a CPU.

In [ ]:
def pad_batch(strings, length):
    '''Right-pad a list of strings into a (B, length) tensor of token ids.'''
    ids = torch.full((len(strings), length), PAD, dtype=torch.long)
    for i, s in enumerate(strings):
        e = encode(s)
        ids[i, :len(e)] = torch.tensor(e)
    return ids

def pretrain(model, corpus, steps=2000, batch_size=128, lr=1e-3, seed=0):
    rng = random.Random(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    losses = []
    for step in tqdm(range(steps), desc="Pretraining"):
        batch = pad_batch(rng.sample(corpus, batch_size), PROMPT_LEN + MAX_NEW)
        logits = model(batch[:, :-1])
        target = batch[:, 1:]
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target.reshape(-1), ignore_index=PAD)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(0)
base = TinyGPT()
pretrain_losses = pretrain(base, corpus)

plt.figure(figsize=(5, 3))
plt.plot(pretrain_losses)
plt.xlabel("step"); plt.ylabel("cross-entropy"); plt.title("Pretraining loss")
plt.show()

## Sampling, and measuring the base model

`sample_completions()` draws completions token by token at temperature 1 (or greedily).
`evaluate()` computes **pass@k**: the fraction of held-out problems for which *at least one* of
$k$ sampled completions is correct.  pass@1 is ordinary sampled accuracy.

Look carefully at the base model's numbers below before you run RL.  Three questions to keep in mind:
1. How does pass@1 compare to the teacher's accuracy?
2. How does pass@16 compare to pass@1?  What does that say about what the model *knows*?
3. How does greedy decoding compare to sampling?

In [ ]:
@torch.no_grad()
def sample_completions(model, prompt_ids, max_new=MAX_NEW, temperature=1.0, greedy=False):
    '''Autoregressively sample completions.  Returns completion ids (B, max_new), PAD after the "."'''
    model.eval()
    B = prompt_ids.size(0)
    seq = prompt_ids.clone()
    done = torch.zeros(B, dtype=torch.bool)
    out = torch.full((B, max_new), PAD, dtype=torch.long)
    for t in range(max_new):
        logits = model(seq)[:, -1, :] / temperature
        logits[:, PAD] = -1e9                      # never sample padding
        nxt = logits.argmax(-1) if greedy else torch.multinomial(F.softmax(logits, -1), 1).squeeze(-1)
        nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
        out[:, t] = nxt
        seq = torch.cat([seq, nxt.unsqueeze(1)], dim=1)
        done |= nxt == EOS
        if done.all():
            break
    model.train()
    return out

def make_eval_set(n=500, seed=12345):
    rng = random.Random(seed)
    return [make_problem(rng) for _ in range(n)]

@torch.no_grad()
def evaluate(model, eval_set, k=1, temperature=1.0, greedy=False):
    '''pass@k: fraction of problems for which at least one of k sampled completions is correct.'''
    prompts = [p for p, _ in eval_set for _ in range(k)]
    prompt_ids = pad_batch(prompts, PROMPT_LEN)
    comp_ids = sample_completions(model, prompt_ids, temperature=temperature, greedy=greedy)
    correct = torch.tensor([reward_fn(p, decode(c)) for p, c in zip(prompts, comp_ids)]).view(-1, k)
    return correct.max(dim=1).values.mean().item()

@torch.no_grad()
def show_samples(model, eval_set, n=8):
    prompts = [p for p, _ in eval_set[:n]]
    comp_ids = sample_completions(model, pad_batch(prompts, PROMPT_LEN))
    for p, c in zip(prompts, comp_ids):
        comp = decode(c)
        print(f"  {p}{comp:<12s} reward={reward_fn(p, comp):.0f}")

def report(model, eval_set, name):
    print(f"{name}:")
    for k in (1, 4, 16):
        print(f"  pass@{k:<2d} = {evaluate(model, eval_set, k=k):.3f}")
    print(f"  greedy  = {evaluate(model, eval_set, k=1, greedy=True):.3f}")

eval_set = make_eval_set()
print("Base model samples (temperature 1):")
show_samples(base, eval_set)
report(base, eval_set, "Base model")

## Stage 2: GRPO

Here is the whole idea of RLVR in one paragraph.  For a prompt $x$, sample a **group** of $G$
completions $y_1, \dots, y_G$ from the current policy $\pi_\theta$, and let the verifier score
each one, $r_i = r(x, y_i)$.  We want to raise the probability of the completions that scored
better than their groupmates and lower the probability of the ones that scored worse.  The
policy-gradient theorem tells us how: the gradient of expected reward is
$\mathbb{E}\left[ A \cdot \nabla_\theta \log \pi_\theta(y \mid x) \right]$, where $A$ is any
*advantage* (reward minus a baseline that does not depend on $y$).

**PPO** learns the baseline with a separate value network.  **GRPO** (Shao et al., 2024) throws
the value network away and uses the group itself as the baseline:

$$A_i = \frac{r_i - \mathrm{mean}(r_{1..G})}{\mathrm{std}(r_{1..G}) + \epsilon}.$$

Because $\log \pi_\theta(y \mid x)$ is a sum over tokens, the loss for one group is

$$\mathcal{L}_{\mathrm{pg}} = -\frac{1}{G}\sum_{i=1}^{G} A_i \cdot \frac{1}{|y_i|}\sum_{t=1}^{|y_i|} \log \pi_\theta(y_{i,t} \mid x, y_{i,<t}).$$

To keep the policy from drifting arbitrarily far from the base model, GRPO adds a KL penalty
with coefficient $\beta$, estimated per token with the "k3" estimator
$\;\mathrm{KL} \approx \exp(\ell_{\mathrm{ref}} - \ell_\theta) - (\ell_{\mathrm{ref}} - \ell_\theta) - 1\;$
where $\ell$ are the token log-probabilities:

$$\mathcal{L} = \mathcal{L}_{\mathrm{pg}} + \beta\,\mathrm{KL}.$$

> **A note on clipping.**  The GRPO paper also multiplies by PPO-style importance ratios
> $\pi_\theta / \pi_{\mathrm{old}}$ and clips them, because it takes several gradient steps on
> each batch of samples.  Here we take exactly **one** gradient step per batch of samples, so the
> ratio is identically 1 and clipping does nothing.  This fully on-policy simplification is what
> nanochat and several other minimal implementations do.

`completion_logprobs()` (given) returns the per-token log-probabilities $\ell_{\theta}$ of each
completion token, shape `(B, L)`.  Padding positions must be masked out of every average.

**TODO:** Implement `group_advantages()` and `grpo_loss()`.

*Hints:*
- Rewards arrive as a flat tensor of length `B*G`, ordered so that each consecutive block of `G` entries shares a prompt.  `rewards.view(-1, group_size)` groups them.
- For `normalize_std=False` (the "Dr. GRPO" variant), skip the division.
- In `grpo_loss`, `mask = (comp_ids != PAD).float()` marks real completion tokens; divide token sums by `mask.sum(1)`.
- Compute the reference log-probabilities inside `torch.no_grad()`; the gradient of the KL term flows only through $\ell_\theta$.
- Return the mean loss over the batch, and the mean KL as a float for logging.

In [ ]:
def completion_logprobs(model, prompt_ids, comp_ids):
    '''Per-token log-probabilities of each completion token under `model`.  Shape (B, L).'''
    seq = torch.cat([prompt_ids, comp_ids], dim=1)
    logits = model(seq[:, :-1])
    logp = F.log_softmax(logits, dim=-1)
    target = seq[:, 1:]
    tok_logp = logp.gather(-1, target.unsqueeze(-1)).squeeze(-1)
    return tok_logp[:, prompt_ids.size(1) - 1:]     # only the completion positions

def group_advantages(rewards, group_size, normalize_std=True, eps=1e-4):
    '''rewards: (B*G,) ordered so each consecutive block of G shares a prompt.  Returns (B*G,) advantages.'''
    # ----- TODO: Student Implementation Starts Here -----
    # The formula, for the G rewards r_1..r_G that share one prompt:
    #     A_i = (r_i - mean(r_1..r_G)) / (std(r_1..r_G) + eps)        (skip the division if normalize_std is False)
    # Steps:
    #   1. r = rewards.view(-1, group_size)                 # shape (B, G): row b holds the G rewards of prompt b
    #   2. subtract the per-row mean                        # r.mean(dim=1, keepdim=True) keeps the (B, 1) shape for broadcasting
    #   3. if normalize_std: divide by the per-row std + eps
    #   4. return the result flattened back to shape (B*G,)
    raise NotImplementedError("Implement group_advantages")
    # ----- TODO: Student Implementation Ends Here -----

def grpo_loss(model, ref_model, prompt_ids, comp_ids, advantages, beta=0.0):
    '''Returns (loss, mean_kl).  loss = policy-gradient term + beta * KL(model || ref_model).'''
    mask = (comp_ids != PAD).float()
    logp = completion_logprobs(model, prompt_ids, comp_ids)
    n_tok = mask.sum(dim=1).clamp(min=1)
    # ----- TODO: Student Implementation Starts Here -----
    # You are given, for a batch of B completions of at most L tokens each:
    #     logp   (B, L)  log pi_theta(y_{i,t} | x, y_{i,<t}) for every completion token   (this carries the gradient)
    #     mask   (B, L)  1.0 for real tokens, 0.0 for padding
    #     n_tok  (B,)    |y_i|, the number of real tokens in completion i
    #     advantages (B,)
    #
    # 1. Policy-gradient term, one number per completion:
    #        pg_i = - A_i * (1/|y_i|) * sum_t logp[i, t]                 (multiply by mask before summing over t)
    # 2. KL term, only when beta > 0 (otherwise kl = zeros like pg):
    #        ref_logp = completion_logprobs(ref_model, prompt_ids, comp_ids)   inside torch.no_grad()
    #        d = ref_logp - logp
    #        kl_i = (1/|y_i|) * sum_t ( exp(d) - d - 1 )                (the k3 estimator; mask the padding again)
    # 3. loss = mean over i of ( pg_i + beta * kl_i )
    # 4. return loss, kl.mean().item()
    raise NotImplementedError("Implement grpo_loss")
    # ----- TODO: Student Implementation Ends Here -----

# Sanity checks for the advantages
r = torch.tensor([1., 0., 0., 1.,   1., 1., 1., 1.,   0., 0., 0., 1.])
adv = group_advantages(r, group_size=4)
assert adv.shape == r.shape
assert torch.allclose(adv.view(3, 4).sum(dim=1), torch.zeros(3), atol=1e-5), "advantages must sum to zero within each group"
assert torch.allclose(adv[4:8], torch.zeros(4)), "a group with identical rewards has zero advantage"
assert adv[0] > 0 and adv[1] < 0 and adv[11] > adv[0], "the lone correct answer in a group of four gets the largest advantage"
adv_nostd = group_advantages(r, group_size=4, normalize_std=False)
assert torch.allclose(adv_nostd[:4], torch.tensor([0.5, -0.5, -0.5, 0.5]))

# Sanity check for the loss: after one small gradient step, a positive-advantage completion becomes
# more likely and a negative-advantage completion becomes less likely
probe = copy.deepcopy(base)
opt = torch.optim.SGD(probe.parameters(), lr=0.01)
p_ids = pad_batch(["3+8+6=", "3+8+6="], PROMPT_LEN)
c_ids = pad_batch(["11+6=17.", "27."], MAX_NEW)
before = completion_logprobs(probe, p_ids, c_ids).detach()
loss, kl = grpo_loss(probe, base, p_ids, c_ids, torch.tensor([1.0, -1.0]), beta=0.1)
loss.backward(); opt.step()
after = completion_logprobs(probe, p_ids, c_ids).detach()
m = (c_ids != PAD)
assert (after[0][m[0]].sum() > before[0][m[0]].sum()) and (after[1][m[1]].sum() < before[1][m[1]].sum())
assert kl >= 0
print("GRPO pieces OK")

## The RLVR training loop

The loop below is the entire algorithm.  Read it once, then run it.  Each step:
1. draws `n_prompts` random problems and repeats each one `group_size` times,
2. samples one completion per copy,
3. scores every completion with the verifier (optionally subtracting a length penalty),
4. computes group-relative advantages,
5. takes one gradient step on the GRPO loss.

We log the mean reward, the fraction of scratchpad-style completions, the mean completion length,
and the KL to the base model, and we periodically evaluate pass@1 on the held-out set.

In [ ]:
def rlvr_train(model, ref_model, steps=300, n_prompts=32, group_size=8, lr=3e-4, beta=0.0,
               normalize_std=True, length_penalty=0.0, reward=None, temperature=1.0,
               seed=0, eval_set=None, eval_every=25, verbose=True):
    reward = reward or reward_fn
    rng = random.Random(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    hist = {"step": [], "reward": [], "scratch_frac": [], "length": [], "kl": [], "eval_step": [], "eval_acc": []}
    pbar = tqdm(range(steps), desc="RLVR", disable=not verbose)
    for step in pbar:
        prompts = [make_problem(rng)[0] for _ in range(n_prompts)]
        prompts = [p for p in prompts for _ in range(group_size)]           # repeat each prompt G times
        prompt_ids = pad_batch(prompts, PROMPT_LEN)
        comp_ids = sample_completions(model, prompt_ids, temperature=temperature)
        comps = [decode(c) for c in comp_ids]
        base_r = torch.tensor([reward(p, c) for p, c in zip(prompts, comps)])
        lengths = (comp_ids != PAD).sum(dim=1).float()
        r = base_r - length_penalty * lengths
        adv = group_advantages(r, group_size, normalize_std)
        loss, kl = grpo_loss(model, ref_model, prompt_ids, comp_ids, adv, beta)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        hist["step"].append(step); hist["reward"].append(base_r.mean().item())
        hist["scratch_frac"].append(sum(is_scratchpad(c) for c in comps) / len(comps))
        hist["length"].append(lengths.mean().item()); hist["kl"].append(kl)
        if eval_set is not None and (step % eval_every == 0 or step == steps - 1):
            hist["eval_step"].append(step); hist["eval_acc"].append(evaluate(model, eval_set, k=1))
        pbar.set_postfix(reward=f"{base_r.mean():.2f}", scratchpad=f"{hist['scratch_frac'][-1]:.2f}")
    return hist

def plot_history(hist, title=""):
    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    axes[0].plot(hist["step"], hist["reward"], label="train reward")
    if hist["eval_step"]:
        axes[0].plot(hist["eval_step"], hist["eval_acc"], "o-", label="held-out pass@1")
    axes[0].set_ylim(0, 1.02); axes[0].legend(); axes[0].set_title("reward")
    axes[1].plot(hist["step"], hist["scratch_frac"]); axes[1].set_ylim(0, 1.02); axes[1].set_title("scratchpad fraction")
    axes[2].plot(hist["step"], hist["length"]); axes[2].set_title("mean completion length")
    axes[3].plot(hist["step"], hist["kl"]); axes[3].set_title("KL to base model")
    for ax in axes:
        ax.set_xlabel("RL step")
    fig.suptitle(title); plt.tight_layout(); plt.show()

def fresh_policy():
    '''A trainable copy of the base model.'''
    return copy.deepcopy(base)

ref_model = copy.deepcopy(base).eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

torch.manual_seed(1)
policy = fresh_policy()
hist = rlvr_train(policy, ref_model, steps=300, beta=0.0, eval_set=eval_set)
plot_history(hist, "GRPO, beta = 0")

print("RL model samples (temperature 1):")
show_samples(policy, eval_set)
report(base, eval_set, "Base model")
report(policy, eval_set, "After RLVR")

### Questions (answer in this cell)

1. **The format shift.**  Nobody told the model to use the scratchpad style, and the reward only looks at the final number.  Why did the scratchpad fraction go to 1?  What in the pretraining data made this possible?

    *Your answer:*

2. **Sharpening versus new capability.**  Compare pass@1, pass@16, and greedy accuracy before and after RL.  Which of these numbers changed, and which did not?  What does that suggest about what RLVR did to the distribution over answers?  (Yue et al. 2025 ask exactly this question about DeepSeek-R1-style training at full scale.)

    *Your answer:*

3. **Where did the remaining errors go?**  Look at the samples after RL.  Are the remaining mistakes in the partial sum or in the final sum?  Why might that be?

    *Your answer:*

# Task for You: experiments

Each experiment re-trains from the same base model with one change.  Each run takes 15 to 30
seconds on a CPU.  Run them, look at the plots, and answer the questions in the cell that follows.

### Experiment A: the KL coefficient $\beta$

The KL term penalizes moving away from the base model.  Try $\beta \in \{0, 0.05, 0.2, 1.0\}$.

In [ ]:
results_beta = {}
for beta in (0.0, 0.05, 0.2, 1.0):
    torch.manual_seed(1)
    pol = fresh_policy()
    h = rlvr_train(pol, ref_model, steps=200, beta=beta, verbose=False)
    results_beta[beta] = dict(reward=sum(h["reward"][-20:]) / 20, scratch=sum(h["scratch_frac"][-20:]) / 20,
                              kl=sum(h["kl"][-20:]) / 20, pass1=evaluate(pol, eval_set, k=1))
    print(f"beta={beta:<5}  reward {results_beta[beta]['reward']:.3f}  scratchpad {results_beta[beta]['scratch']:.2f}  "
          f"KL {results_beta[beta]['kl']:.3f}  held-out pass@1 {results_beta[beta]['pass1']:.3f}")

### Experiment B: a length penalty

Reasoning models are expensive because they write a lot.  Subtract `length_penalty` times the
number of tokens from every reward and try `length_penalty` $\in \{0, 0.02, 0.05, 0.1\}$.
Remember that a scratchpad completion is about 8 tokens and a direct one about 3.

In [ ]:
results_len = {}
for lp in (0.0, 0.02, 0.05, 0.1):
    torch.manual_seed(1)
    pol = fresh_policy()
    h = rlvr_train(pol, ref_model, steps=200, length_penalty=lp, verbose=False)
    results_len[lp] = dict(reward=sum(h["reward"][-20:]) / 20, scratch=sum(h["scratch_frac"][-20:]) / 20,
                           length=sum(h["length"][-20:]) / 20, pass1=evaluate(pol, eval_set, k=1))
    print(f"length_penalty={lp:<5}  reward {results_len[lp]['reward']:.3f}  scratchpad {results_len[lp]['scratch']:.2f}  "
          f"length {results_len[lp]['length']:.1f}  held-out pass@1 {results_len[lp]['pass1']:.3f}")

### Experiment C: reward hacking with a sloppy verifier

Suppose a careless engineer writes the verifier as "the correct number appears *somewhere* in the
completion."  Train against `loose_reward` and then measure the model with the *real* verifier.
Print some samples and look at what the model learned to do.

In [ ]:
def loose_reward(prompt, completion):
    '''A sloppy verifier: 1.0 if the correct answer appears anywhere before the period.'''
    a, b, c = (int(x) for x in prompt[:-1].split("+"))
    return 1.0 if str(a + b + c) in completion.split(".")[0] else 0.0

torch.manual_seed(1)
hacked = fresh_policy()
h = rlvr_train(hacked, ref_model, steps=200, reward=loose_reward, verbose=False)
plot_history(h, "training against the loose verifier")
print(f"loose reward at end of training: {sum(h['reward'][-20:]) / 20:.3f}")
print(f"TRUE held-out pass@1:            {evaluate(hacked, eval_set, k=1):.3f}")
print("samples:")
show_samples(hacked, eval_set, n=10)

### Questions on the experiments (answer in this cell)

**A.** As $\beta$ grows, what happens to the scratchpad fraction and to pass@1?  Why would anyone
want a nonzero $\beta$ in practice, given that it only hurts here?  (Think about what the reward
does *not* measure.)

*Your answer:*

**B.** Even a length penalty of 0.02 per token, tiny compared with a reward of 1, changes the
model's behavior a lot.  Work out the expected reward of the two styles under the base model with
`length_penalty=0.02` and explain the outcome.  What does this say about the response-length
growth reported in the DeepSeek-R1 paper?

*Your answer:*

**C.** Describe concretely how the model exploited the loose verifier.  Why did the true accuracy
get *worse* than the base model's?  Name one real-world verifier (for math, code, or anything else)
and one way it could be gamed in the same spirit.

*Your answer:*

# Extra Material for Help and Reference

## Where this sits in the landscape

| Method | Baseline for the advantage | Needs a reward model? | Where you saw it |
|---|---|---|---|
| REINFORCE | none, or a running mean | no | this notebook with `normalize_std=False`, `G` large |
| PPO (used in RLHF, Ouyang et al. 2022) | a learned value network | yes, for RLHF | lecture on RLHF |
| GRPO (Shao et al. 2024; DeepSeek-R1) | the mean of a group of samples | no: the reward is a program | this notebook |
| Dr. GRPO (Liu et al. 2025) | group mean, **no** std normalization, no length normalization | no | `normalize_std=False` |

**Why "verifiable"?**  RLHF learns a reward model from human preferences, and the policy can
learn to fool it.  RLVR replaces the reward model with a checker: an answer key, a unit test, a
proof assistant.  The reward is exact, but only for tasks that have a checker, and, as Experiment C
shows, only as good as the checker.

**What actually happens during RLVR.**  In our toy, RL never taught the model anything it could
not already do; it made the model *reliably* do what it already did *sometimes*.  Whether the same
is true at scale is an active debate: Yue et al. (2025) argue that pass@k at large $k$ barely
improves after RL on real models, while others find genuinely new behaviors with enough
training.  Our experiments give you a clean setting to think about what evidence would settle it.

## Readings
- Shao et al. 2024, *DeepSeekMath*, Section 4 introduces GRPO: <https://papers.baulab.info/papers/also/Shao-2024.pdf>
- DeepSeek-AI 2025, *DeepSeek-R1*: <https://papers.baulab.info/papers/DeepSeek-2025.pdf>
- Lambert et al. 2025, *Tülu 3*, which coined the term RLVR: <https://papers.baulab.info/papers/also/Lambert-2025.pdf>
- Yue et al. 2025, *Does RL really incentivize reasoning capacity beyond the base model?* (the pass@k argument)
- Liu et al. 2025, *Understanding R1-Zero-like training: a critical perspective* (Dr. GRPO)
- Karpathy's nanochat `chat_rl` and McGill's nano-aha-moment are single-file GRPO implementations at real-model scale.

## Credits

The algorithm in this notebook is GRPO from Shao et al. (2024), as used to train DeepSeek-R1
(DeepSeek-AI, 2025); the name "RLVR" is from Lambert et al. (2025).  The per-token KL estimator
$\exp(d) - d - 1$ is John Schulman's "k3" estimator (blog post *Approximating KL Divergence*,
2020), which GRPO adopted.  Taking a single on-policy gradient step per batch of samples, so
that PPO's importance ratios and clipping disappear, follows Andrej Karpathy's nanochat (2025).
The `normalize_std=False` option is the Dr. GRPO variant of Liu et al. (2025).  The pass@k
framing of what RL does to a model follows Yue et al. (2025).  The structure of the loop was
informed by three single-file implementations: McGill NLP's nano-aha-moment (Kazemnejad et al.,
2025), GRPO-Zero, and superlinear-ai's microGRPO.  The toy problem itself, the noisy teacher with
two solution styles, and the sloppy-verifier experiment were designed for CS 7150 (Fall 2026).